#### ENRICH `ORDER_ITEMS` FROM BRONZE TO SILVER

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDER_ITEMS_ENRICHED'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
SOURCE = f"{CATALOG}.BRONZE.ORDER_ITEMS"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
order_items_df = spark.table(SOURCE)
order_items_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

# Select and cast columns properly
order_items_enriched_df = (
    order_items_df
    .select(
        trim(col("order_id")).alias("order_id"),
        col("order_item_id").cast("int").alias("order_item_id"),
        trim(col("product_id")).alias("product_id"),
        trim(col("seller_id")).alias("seller_id"),
        to_timestamp(col("shipping_limit_date")).alias("shipping_limit_date"),
        col("price").cast("double").alias("price"),
        col("freight_value").cast("double").alias("freight_value"),
        col("file_path")
        
)
)
# Show one row for verification
order_items_enriched_df.limit(1).display()

In [0]:
from pyspark.sql.window import Window

w = (
    Window.partitionBy("order_id", "order_item_id")
    .orderBy(col("shipping_limit_date").desc_nulls_last(), col("price").desc_nulls_last())
)

nodup_df = order_items_enriched_df.withColumn("rn", row_number().over(w)).where(col("rn") == 1).drop(col("rn"))

nodup_df.limit(1).display()

In [0]:
flagged_df = (nodup_df.select("order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value", "file_path")
                      .withColumn("is_price_non_positive", col("price") <= 0)
                      .withColumn("is_freight_negative", col("freight_value") <= 0)
                      .withColumn("is_ship_date_outlier", (year(col("shipping_limit_date")) < lit(2017)) | (year(col("shipping_limit_date")) > lit(2019)))
                      )

flagged_df.limit(1).display()

In [0]:
order_items_trans_df = (
    flagged_df
    .withColumn("total_item_value", round(col("price") + col("freight_value")))
    .withColumn(
        "freight_ratio", 
        when(col("price") > 0, round(col("freight_value") / col("price"), 3)).otherwise(lit(None))
    )
    .withColumn("ship_year", year(col("shipping_limit_date")))
    .withColumn("ship_month", month(col("shipping_limit_date")))
    .withColumn("ship_date", dayofmonth(col("shipping_limit_date")))
)
order_items_trans_df.limit(1).display()

In [0]:
order_items_enriched_df = (order_items_trans_df
                           .select("order_id", "order_item_id", "product_id", "seller_id", 
                                   "shipping_limit_date", "price", "freight_value", "is_price_non_positive", "is_freight_negative", "is_ship_date_outlier", "total_item_value", "freight_ratio", "ship_year", "ship_month", "ship_date", "file_path")
                           ).withColumn("load_timestamp", current_timestamp())

order_items_enriched_df.limit(1).display()

In [0]:
order_items_enriched_df.createOrReplaceTempView('enriched_order_items_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED`

In [0]:
def write_audit(run_id, status, start_time, rows=None, error=None):
  end = datetime.now()
  row = (run_id, "ORDER_ITEMS_ENRICHED_LOAD", "SILVER", SOURCE, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
  (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
  result = spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
                      USING (
                        SELECT *
                        FROM (
                          SELECT *,
                                ROW_NUMBER() OVER (
                                    PARTITION BY order_id
                                    ORDER BY load_timestamp DESC
                                ) rn
                          FROM enriched_order_items_temp_vw
                        )
                        WHERE rn = 1
                      ) src
                      ON (src.order_id = tgt.order_id)
                      WHEN MATCHED
                          THEN UPDATE SET
                          tgt.order_item_id = src.order_item_id,
                          tgt.product_id = src.product_id,
                          tgt.seller_id = src.seller_id,
                          tgt.shipping_limit_date = src.shipping_limit_date
                      WHEN NOT MATCHED THEN
                      INSERT (order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value, is_price_non_positive,
                      is_freight_negative, is_ship_date_outlier, total_item_value, freight_ratio, ship_year, ship_month, ship_date, origin_file_path,
                      load_timestamp)
                      VALUES (src.order_id, src.order_item_id, src.product_id, src.seller_id, src.shipping_limit_date, src.price, src.freight_value,
                      src.is_price_non_positive, src.is_freight_negative, src.is_ship_date_outlier, src.total_item_value, src.freight_ratio, 
                      src.ship_year, src.ship_month, src.ship_date, src.file_path, current_timestamp)
                      ''')
  m = result.first().asDict()
  inserted = m.get("num_inserted_rows") or 0
  updated  = m.get("num_updated_rows") or 0
  rows_written = int(inserted + updated)
  print(f"inserted={inserted}, updated={updated}")

except Exception:
  status, err = "FAILED", traceback.format_exc()
  raise
finally:
  write_audit(run_id, status, start_time, rows_written, err)   

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED')